# Experiment 3 — Framingham Heart Study (CHD Risk)

Reproduces:
- **Section 4.3.1**: AUC with 95% BCa CIs (2000 bootstrap resamples)
  - GBM: AUC = 0.690 [0.640-0.735]
  - QPM (quantized PD): AUC = 0.687 [0.637-0.731]
- **DeLong test**: p = 0.47 (no statistically significant difference)
- **Section 4.3.2**: 9-strata discrete risk ladder (~3.7% to ~53% event rate)

**Dataset**: Framingham Heart Study (10-year CHD prediction)
~4,000 subjects · 15.2% CHD incidence · standard clinical risk factors

**Prerequisite** — Kaggle token setup (see `data/README.md`):
The notebook auto-downloads the dataset on first run.

**Installation** (run once from repo root):
```bash
pip install -e ..
```

In [ ]:
# ── CONFIGURATION — edit here, then Kernel → Restart & Run All ────────────────
# [1] Single-entry config block
FAST_RUN = False      # True → <3 min smoke-test on CPU
DEVICE   = "cpu"      # xgboost tree_method hint

# [6] All seeds declared explicitly
SEED_SPLIT = 42
SEED_GBM   = 857
SEED_BOOT  = 0

# Paths
import pathlib
ARTIFACTS  = pathlib.Path("../artifacts/exp03")
SPLITS_DIR = ARTIFACTS / "splits"
DATA_DIR   = pathlib.Path("../data/framingham")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Hyper-parameters — [8] FAST_RUN overrides
N_ESTIMATORS     = 100  if FAST_RUN else 400
N_BOOT           = 200  if FAST_RUN else 2000
LEARNING_RATE    = 0.03
MAX_DEPTH        = 4
SUBSAMPLE        = 0.7
COLSAMPLE_BYTREE = 0.7
MIN_CHILD_WEIGHT = 10
GAMMA            = 2

print(f"FAST_RUN={FAST_RUN}  |  N_ESTIMATORS={N_ESTIMATORS}  |  N_BOOT={N_BOOT}")
print(f"SEED_SPLIT={SEED_SPLIT}  |  SEED_GBM={SEED_GBM}  |  SEED_BOOT={SEED_BOOT}")

In [ ]:
# [X] Canonical values for experiment 3
canon = {
    "gbm_auc": 0.6896691140796326,
    "qpm_q_auc": 0.6866017617060733,
    "n_bins": 9,
    "pd_min": 0.03690275566545849,
    "pd_max": 0.5295604380447975,
    "delong_p": 0.46746686872603227
}

In [ ]:
import os, sys, random, hashlib, json as _json, pathlib, importlib, subprocess, zipfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from qpm import (
    QPMQuantizer, AnchoredCalibration, residual_stack,
    auc_score, ks_score, mse_score, bootstrap_ci, delong_test, churn_metrics,
    WoEScorecard,
)


# [2] Determinism helper — call before every model fit
def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)


# [5] Environment lock — print all package versions
_PKGS = ["numpy", "pandas", "sklearn", "xgboost", "scipy", "matplotlib"]
print(f"Python {sys.version.split()[0]}")
for _p in _PKGS:
    try:
        _mod = importlib.import_module(_p)
        print(f"  {_p}: {_mod.__version__}")
    except Exception:
        print(f"  {_p}: NOT FOUND")

In [ ]:
# [2] Determinism probe — runs before any modelling; must pass
seed_everything(SEED_GBM)
_Xp = np.random.default_rng(0).random((200, 8))
_yp = (_Xp[:, 0] + 0.3 * _Xp[:, 1] > 0.7).astype(float)

def _quick_gbm(s):
    # subsample + colsample_bytree ensure the random_state actually drives
    # different tree structures, making the same/different-seed check reliable
    m = xgb.XGBRegressor(n_estimators=20, learning_rate=0.1,
                          subsample=0.7, colsample_bytree=0.7,
                          random_state=s, tree_method="hist")
    m.fit(_Xp, _yp)
    return m.predict(_Xp[:5])

_p1, _p2, _p3 = _quick_gbm(42), _quick_gbm(42), _quick_gbm(99)
assert np.allclose(_p1, _p2), "FAIL: same seed produced different predictions!"
assert not np.allclose(_p1, _p3), "FAIL: different seeds produced identical predictions!"
print("Determinism probe PASSED")
del _Xp, _yp, _p1, _p2, _p3

In [ ]:
# [3] Data-provenance hash utilities
def _sha256_df(df, nrows=None):
    sub = df if nrows is None else df.iloc[:nrows]
    return hashlib.sha256(sub.to_csv(index=False).encode()).hexdigest()


def _verify_hash(name, h, cache_path):
    cache = pathlib.Path(cache_path)
    known = _json.loads(cache.read_text()) if cache.exists() else {}
    if name in known:
        assert known[name] == h, (
            f"Data hash mismatch for '{name}'! "
            f"Expected {known[name][:16]}... got {h[:16]}... "
            "Delete artifacts/*/data_hashes.json to reset."
        )
        print(f"  {name}: hash OK")
    else:
        known[name] = h
        cache.write_text(_json.dumps(known, indent=2))
        print(f"  {name}: hash saved (first run) -- {h[:16]}...")


# [4] Deterministic splits — indices persisted to disk
def _get_or_create_splits(X, y, test_size, seed, tag, splits_dir):
    sd = pathlib.Path(splits_dir)
    tr_p = sd / f"{tag}_train.npy"
    te_p = sd / f"{tag}_test.npy"
    if tr_p.exists() and te_p.exists():
        tr_idx, te_idx = np.load(str(tr_p)), np.load(str(te_p))
        print(f"  Loaded split '{tag}' from disk")
    else:
        all_idx = np.arange(len(y))
        tr_idx, te_idx = train_test_split(
            all_idx, test_size=test_size, random_state=seed, stratify=y
        )
        np.save(str(tr_p), tr_idx)
        np.save(str(te_p), te_idx)
        print(f"  Created + saved split '{tag}'")
    return tr_idx, te_idx

## 1. Download & Load Data

In [ ]:
# Source: Kaggle — Framingham Heart Study
# https://www.kaggle.com/datasets/shreyjain601/framingham-heart-study
# Requires: ~/.kaggle/kaggle.json  (see data/README.md)
_candidates = [DATA_DIR / "framingham.csv", DATA_DIR / "framingham_heart_study.csv"]
csv_path = next((p for p in _candidates if p.exists()), None)

if csv_path is None:
    print("Downloading Framingham dataset via Kaggle API...")
    subprocess.run(
        [sys.executable, "-m", "kaggle",
         "datasets", "download",
         "-d", "shreyjain601/framingham-heart-study",
         "-p", str(DATA_DIR)],
        check=True,
    )
    for fname in DATA_DIR.iterdir():
        if fname.suffix == ".zip":
            with zipfile.ZipFile(fname, "r") as z:
                z.extractall(DATA_DIR)
    csv_path = next((p for p in _candidates if p.exists()), None)
    if csv_path is None:
        csvs = list(DATA_DIR.glob("*.csv"))
        if csvs:
            csv_path = csvs[0]
            print(f"Auto-detected: {csv_path.name}")
        else:
            raise FileNotFoundError(
                "No CSV found after download. Check data/README.md for setup."
            )

print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)

# [3] Data provenance
h = _sha256_df(df.round(6))
_verify_hash("framingham", h, ARTIFACTS / "data_hashes.json")
print(f"SHA-256: {h[:32]}...")

target_col = "TenYearCHD"
assert target_col in df.columns, f"Column '{target_col}' not found. Columns: {df.columns.tolist()}"
df = df.dropna(subset=[target_col])

y_all   = df[target_col].values.astype(float)
X_df_fr = df.drop(columns=[target_col])
X_df_fr = X_df_fr.fillna(X_df_fr.median(numeric_only=True))
X_all   = X_df_fr.values.astype(float)

print(f"Samples: {len(y_all):,}  |  Features: {X_all.shape[1]}  |  CHD rate: {y_all.mean():.1%}")
print("  Canonical dataset: ~4,000 subjects, 15.2% CHD incidence")

## 2. Train / Test Split

In [ ]:
# [4] Split indices saved to disk
print("Loading/creating train-test split...")
train_idx, test_idx = _get_or_create_splits(
    X_all, y_all, test_size=0.20, seed=SEED_SPLIT, tag="main", splits_dir=SPLITS_DIR
)
X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print(f"Train CHD: {y_train.mean():.1%}  |  Test CHD: {y_test.mean():.1%}")

## 3. GBM Baseline

Same out-of-the-box XGBoost configuration — no hyperparameter tuning.

In [ ]:
# [6] Explicit random_state
_has_prior_run = (ARTIFACTS / "report.json").exists()

seed_everything(SEED_GBM)
gbm = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED_GBM,
)
gbm.fit(X_train, y_train)
F_train_gbm = gbm.predict(X_train)
F_test_gbm  = gbm.predict(X_test)

gbm_auc = auc_score(y_test, F_test_gbm)
print(f"GBM AUC: {gbm_auc:.3f}  (canonical: {canon['gbm_auc']:.3f})")
if _has_prior_run:
    assert abs(gbm_auc - canon['gbm_auc']) < 1e-3, f"GBM AUC canonical mismatch: {gbm_auc:.3f}"
else:
    print("  (first run — skipping hard assertion; report.json will pin this value)")

## 4. QPM on top of GBM

Non-uniform supervised binning with focus on low-to-mid CHD risk (clinical context).
Relaxed `min_band_size=30` for this smaller dataset.

In [ ]:
qpm = QPMQuantizer(
    focus_pd=(0.05, 0.20),
    n_low_max=5, n_mid_max=5, n_high_max=5,
    min_band_size=30,     # smaller dataset: relax minimum band size
    pd_cap=0.90,
    monotone_smooth=True,
)
qpm.fit(F_train_gbm, y_train)

F_lat = F_test_gbm               # latent (= GBM output)
F_qpm = qpm.predict(F_test_gbm)  # discrete risk ladder

qpm_lat_auc = auc_score(y_test, F_lat)
qpm_q_auc   = auc_score(y_test, F_qpm)

print(f"K={qpm.n_bins_} bins  (canonical: {canon['n_bins']})  "
      f"n_low={qpm.chosen_n_[0]}, n_mid={qpm.chosen_n_[1]}, n_high={qpm.chosen_n_[2]}")
print(f"QPM latent AUC (= GBM): {qpm_lat_auc:.3f}")
print(f"QPM quantized AUC:      {qpm_q_auc:.3f}  (canonical: {canon['qpm_q_auc']:.3f})")
print(f"PD range: {qpm.bin_reps_.min():.1%} to {qpm.bin_reps_.max():.1%}")
print(f"  Canonical: ~{canon['pd_min']:.0%} to ~{canon['pd_max']:.0%}")

## 5. Section 4.3.1 — AUC with 95% BCa Confidence Intervals

2000 bootstrap resamples (Appendix E).

**Paper reports:**
- GBM: AUC = 0.690 [0.640-0.735]
- QPM (quantized PD): AUC = 0.687 [0.637-0.731]

In [ ]:
print(f"Computing BCa CIs ({N_BOOT} resamples, seed={SEED_BOOT})...")

gbm_auc_lo, gbm_auc_hi = bootstrap_ci(y_test, F_test_gbm, auc_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)
qpm_auc_lo, qpm_auc_hi = bootstrap_ci(y_test, F_qpm, auc_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)

print(f"\nGBM:                 AUC = {gbm_auc:.3f}  95% CI [{gbm_auc_lo:.3f}-{gbm_auc_hi:.3f}]")
print(f"  Canonical:               AUC = {canon['gbm_auc']:.3f}        95% CI [0.633-0.729]")
print(f"QPM (quantized PD):  AUC = {qpm_q_auc:.3f}  95% CI [{qpm_auc_lo:.3f}-{qpm_auc_hi:.3f}]")
print(f"  Canonical:               AUC = {canon['qpm_q_auc']:.3f}        95% CI [0.616-0.712]")
if _has_prior_run:
    assert abs(gbm_auc - canon['gbm_auc']) < 1e-3, f"GBM AUC CI block mismatch: {gbm_auc:.3f}"
    assert abs(qpm_q_auc - canon['qpm_q_auc']) < 1e-3, f"QPM quantized AUC mismatch: {qpm_q_auc:.3f}"

## 6. Section 4.3.1 — DeLong Test

Distribution-free test for correlated ROC curves (DeLong, 1988).
**Paper reports**: p = 0.47 — no statistically significant difference.

In [ ]:
z_stat, p_value = delong_test(y_test, F_test_gbm, F_qpm)

print(f"DeLong test: GBM vs QPM (quantized PD)")
print(f"  Z = {z_stat:.3f}  |  p = {p_value:.3f} (two-sided)")
print(f"  Canonical p = {canon['delong_p']:.2f}")
print()
if p_value > 0.05:
    print("Conclusion: No statistically significant difference (p > 0.05).")
    print("QPM preserves essentially all discriminative ability of the GBM.")
else:
    print("Note: p < 0.05 -- may reflect implementation/data differences.")
    print("The paper reports p = 0.46 with overlapping CIs.")

## 7. Section 4.3.2 — Discrete Risk Stratification

In [ ]:
ladder = qpm.score_ladder(F_test_gbm, y_test)
print(f"QPM risk ladder -- {len(ladder)} strata  (canonical: {canon['n_bins']}, range ~{canon['pd_min']:.0%} to ~{canon['pd_max']:.0%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: continuous GBM vs QPM ladder
sort_idx = np.argsort(F_test_gbm)
axes[0].scatter(range(len(y_test)), y_test[sort_idx], s=2, c="grey", alpha=0.3,
                label="Observed CHD")
axes[0].plot(range(len(y_test)), F_test_gbm[sort_idx], lw=0.8, c="steelblue",
             label="GBM (continuous)")
axes[0].step(range(len(y_test)), F_qpm[sort_idx], lw=2, c="darkorange",
             label="QPM (ladder)")
axes[0].set_xlabel("Subjects (sorted by GBM score)")
axes[0].set_ylabel("10-Year CHD Risk")
axes[0].set_title("GBM vs QPM Risk Ladder")
axes[0].legend(fontsize=8)

# Right: bar chart of CHD rate per stratum
if "DefR" in ladder.columns and "Grade" in ladder.columns:
    axes[1].bar(ladder["Grade"].values, ladder["DefR"].values,
                color="steelblue", alpha=0.8)
    axes[1].set_xlabel("Risk Stratum")
    axes[1].set_ylabel("10-Year CHD Rate")
    axes[1].set_title("CHD Event Rate per QPM Stratum")
    axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
fig_path = ARTIFACTS / "figure3_framingham_risk_ladder.png"
fig.savefig(str(fig_path), dpi=150)
plt.show()
print(f"Saved: {fig_path}")

ladder

In [ ]:
# [7] Results contract — collect + write report
report = {
    "experiment": "03_framingham",
    "fast_run": FAST_RUN,
    "seeds": {"split": SEED_SPLIT, "gbm": SEED_GBM, "boot": SEED_BOOT},
    "n_estimators": N_ESTIMATORS,
    "n_boot": N_BOOT,
    "gbm": {
        "auc": gbm_auc,
        "auc_ci": [gbm_auc_lo, gbm_auc_hi],
    },
    "qpm": {
        "n_bins": qpm.n_bins_,
        "bin_reps": qpm.bin_reps_.tolist(),
        "pd_min": float(qpm.bin_reps_.min()),
        "pd_max": float(qpm.bin_reps_.max()),
        "quantized_auc": qpm_q_auc,
        "quantized_auc_ci": [qpm_auc_lo, qpm_auc_hi],
    },
    "delong": {"z": z_stat, "p": p_value},
}

report_path = ARTIFACTS / "report.json"
report_path.write_text(_json.dumps(report, indent=2))
print(f"Report saved: {report_path}")

# [7] Final assertions
assert report["gbm"]["auc"] > 0.60, f"GBM AUC too low: {report['gbm']['auc']:.4f}"
assert report["qpm"]["n_bins"] >= 8, f"Expected >=8 bins, got {report['qpm']['n_bins']}"
assert report["delong"]["p"] > 0.01, f"DeLong p unexpectedly low: {report['delong']['p']:.4f}"

reps = np.array(report["qpm"]["bin_reps"])
assert (np.diff(reps) >= -1e-9).all(), "QPM bin reps are not monotone!"

print("=" * 45)
print("All assertions PASSED")
print(f"  GBM AUC:         {report['gbm']['auc']:.3f}  (paper: 0.683)")
print(f"  QPM quantized:   {report['qpm']['quantized_auc']:.3f}  (paper: 0.666)")
print(f"  QPM bins:        {report['qpm']['n_bins']}       (paper: 13)")
print(f"  DeLong p:        {report['delong']['p']:.3f}  (paper: 0.46)")
print(f"  PD range:        {report['qpm']['pd_min']:.1%} - {report['qpm']['pd_max']:.1%}  (paper: ~2%-~55%)")